# General SOM Data Preprocessing Notebook

By: Ty Janoski

A unified preprocessing notebook for all ERA5 variables used in the NYC flash flood SOM analysis.

Supported variables: Z500, MSLP, TCWV, IVT, TP

## Configuration

Set `VARIABLE` to the desired ERA5 variable and `MAKE_PLOTS` to toggle example visualizations.

In [ ]:
# ---------- USER CONFIGURATION ----------
VARIABLE = "Z500"  # Options: "Z500", "mslp", "tcwv", "ivt", "tp"
MAKE_PLOTS = True
# ----------------------------------------

## Variable Definitions

In [ ]:
import numpy as np

CONFIGS = {
    "Z500": {
        "file_path": "/mnt/drive2/ERA5/NC_files/combined/era5_Z500_hourly_warm_season_US.nc",
        "load_func": "dataarray",
        "time_dim": "time",
        "lat_dim": "lat",
        "lon_dim": "lon",
        "compute_anomalies": True,
        "compute_magnitude": None,
        "output_prefix": "era5_Z500",
        "save_daily": True,
        "plot_config": {
            "title": "Z$_{500}$",
            "cmap": "viridis",
            "raw_levels": np.arange(522, 595, 6),
            "raw_transform": lambda x: x / 10 / 9.81,
            "raw_unit": "dam",
            "std_levels": np.arange(-3, 3.1, 0.5),
            "extend": "neither",
            "fig_dir": "figs/Z500-SOM",
        },
    },
    "mslp": {
        "file_path": "/mnt/drive2/ERA5/NC_files/combined/era5_mslp_hourly_warm_season_US.nc",
        "load_func": "dataarray",
        "time_dim": "valid_time",
        "lat_dim": "latitude",
        "lon_dim": "longitude",
        "compute_anomalies": True,
        "compute_magnitude": None,
        "output_prefix": "era5_mslp",
        "save_daily": True,
        "plot_config": {
            "title": "MSLP",
            "cmap": "viridis",
            "raw_levels": np.arange(988, 1032, 4),
            "raw_transform": lambda x: x / 100,
            "raw_unit": "mb",
            "std_levels": np.arange(-3, 3.1, 0.5),
            "extend": "both",
            "fig_dir": "../figs/mslp-SOM",
        },
    },
    "tcwv": {
        "file_path": "/mnt/drive2/ERA5/NC_files/combined/era5_tcwv_hourly_warm_season_US.nc",
        "load_func": "dataarray",
        "time_dim": "time",
        "lat_dim": "lat",
        "lon_dim": "lon",
        "compute_anomalies": True,
        "compute_magnitude": None,
        "output_prefix": "era5_tcwv",
        "save_daily": True,
        "plot_config": {
            "title": "TCWV",
            "cmap": "viridis",
            "raw_levels": np.arange(0, 51, 5),
            "raw_transform": None,
            "raw_unit": "kg m$^{-2}$",
            "std_levels": np.arange(-3, 3.1, 0.5),
            "extend": "max",
            "fig_dir": "figs/tcwv-SOM",
        },
    },
    "ivt": {
        "file_path": "/mnt/drive2/ERA5/NC_files/combined/era5_viwv_hourly_warm_season_US.nc",
        "load_func": "dataset",
        "time_dim": "valid_time",
        "lat_dim": "latitude",
        "lon_dim": "longitude",
        "compute_anomalies": True,
        "compute_magnitude": {
            "components": ["viwve", "viwvn"],
            "name": "ivt",
        },
        "output_prefix": "era5_ivt",
        "save_daily": True,
        "plot_config": {
            "title": "IVT",
            "cmap": "viridis",
            "raw_levels": np.arange(0, 701, 50),
            "raw_transform": None,
            "raw_unit": "kg m$^{-1}$ s$^{-1}$",
            "std_levels": np.arange(-6, 6.1, 1.0),
            "extend": "both",
            "fig_dir": "../figs/Z500-and-ivtxy-SOM",
            "components": [
                ("ivt", "IVT$_{mag}$", np.arange(0, 701, 50), "max"),
                ("viwve", "IVT$_{x}$", np.arange(-700, 701, 100), "both"),
                ("viwvn", "IVT$_{y}$", np.arange(-700, 701, 100), "both"),
            ],
        },
    },
    "tp": {
        "file_path": "/mnt/drive2/ERA5/NC_files/hourly_sliced/era5_tp_*.nc",
        "load_func": "mfdataset",
        "load_kwargs": {"combine": "by_coords", "decode_times": True, "chunks": {"valid_time": 8760}},
        "var_name": "tp",
        "time_dim": "valid_time",
        "lat_dim": "latitude",
        "lon_dim": "longitude",
        "compute_anomalies": False,
        "compute_magnitude": None,
        "output_prefix": "era5_tp",
        "save_daily": False,
        "plot_config": None,
    },
}

config = CONFIGS[VARIABLE]
print(f"Processing: {VARIABLE}")

## Setup

### Imports

In [ ]:
import cartopy.crs as ccrs
import cmweather  # noqa: F401
import matplotlib.pyplot as plt
import pandas as pd
import scienceplots  # noqa: F401
import xarray as xr
from dask.diagnostics.progress import ProgressBar  # noqa: F401

plt.style.use(["science", "nature", "grid"])
plt.rcParams["text.usetex"] = True

### Flash Flood Event Times

In [ ]:
# Read in CSV file with flash flood events
df = pd.read_csv("../data/storm_data_search_results.csv")

# Remove rows where EVENT_ID is not a digit
df = df[df["EVENT_ID"].astype(str).str.isdigit()]

# Turn BEGIN_TIME and END_TIME into strings with leading zeros if necessary
df["BEGIN_TIME"] = df["BEGIN_TIME"].fillna(0).astype(int).astype(str).str.zfill(4)
df["END_TIME"] = df["END_TIME"].fillna(0).astype(int).astype(str).str.zfill(4)

# Combine TIME and DATE into a single datetime string
begin_str = df["BEGIN_DATE"] + " " + df["BEGIN_TIME"]  # type: ignore
end_str = df["END_DATE"] + " " + df["END_TIME"]  # type: ignore

# Convert the datetime strings to pandas datetime objects
df["BEGIN_DATETIME"] = pd.to_datetime(
    begin_str, format="%m/%d/%Y %H%M", errors="coerce"
)
df["END_DATETIME"] = pd.to_datetime(end_str, format="%m/%d/%Y %H%M", errors="coerce")

# Take only the first row for each EPISODE_ID
df_unique = df.drop_duplicates(subset=["EPISODE_ID"], keep="first").copy()

# Create a list of datetimes rounded to the nearest preceding hour
event_hours = df_unique["BEGIN_DATETIME"].dt.floor("h").tolist()  # type: ignore

## ERA5 Data Loading

In [ ]:
# Load ERA5 data based on the configured load function
if config["load_func"] == "dataarray":
    data = xr.load_dataarray(config["file_path"], decode_timedelta=True)
elif config["load_func"] == "dataset":
    data = xr.load_dataset(config["file_path"], decode_timedelta=True)
elif config["load_func"] == "mfdataset":
    data = xr.open_mfdataset(config["file_path"], **config["load_kwargs"])
    with ProgressBar():
        data = data.load()

# Compute magnitude variable if needed (e.g., IVT from components)
if config["compute_magnitude"] is not None:
    mag_cfg = config["compute_magnitude"]
    data[mag_cfg["name"]] = np.sqrt(
        sum(data[c] ** 2 for c in mag_cfg["components"])
    )

print(data)

## Standardized Anomaly Computation

In [ ]:
time_dim = config["time_dim"]
lat_dim = config["lat_dim"]

if config["compute_anomalies"]:
    # Group by day of year to calculate mean and std dev
    mean_doy = data.groupby(f"{time_dim}.dayofyear").mean(dim=time_dim)
    std_doy = data.groupby(f"{time_dim}.dayofyear").std(dim=time_dim)

    std_doy_smooth = (
        std_doy
        .sortby("dayofyear")
        .rolling(dayofyear=14, center=True, min_periods=7)
        .mean()
    )

    # Calculate standardized anomalies
    anoms = data.groupby(f"{time_dim}.dayofyear") - mean_doy
    norm = anoms.groupby(f"{time_dim}.dayofyear") / std_doy_smooth

    # Weight by square root of cosine of latitude
    weights = np.sqrt(np.cos(np.deg2rad(norm[lat_dim])))
    norm_weighted = norm * weights

    print("Standardized anomalies computed.")
else:
    norm = None
    norm_weighted = None
    print("Anomaly computation skipped for this variable.")

## Example Visualization

In [ ]:
if MAKE_PLOTS and config["plot_config"] is not None and config["compute_anomalies"]:
    pcfg = config["plot_config"]
    lon_dim = config["lon_dim"]

    # Handle IVT multi-component plots
    if "components" in pcfg:
        components = pcfg["components"]
        nrows = len(components)
        fig, axs = plt.subplots(
            nrows, 3, figsize=(6.5, 2 * nrows),
            subplot_kw={"projection": ccrs.PlateCarree()}, dpi=600,
        )

        datasets = [
            (data, ""),
            (norm, " Standardized Anomalies"),
            (norm_weighted, " Standardized Anomalies Weighted"),
        ]

        for row, (var, label, raw_levels, extend) in enumerate(components):
            for col, (dset, suffix) in enumerate(datasets):
                ax = axs[row, col]
                plot_data = dset[var].isel({time_dim: 0})
                levels = raw_levels if col == 0 else pcfg["std_levels"]
                cmap = "viridis" if (row == 0 and col == 0) else "balance"

                c = ax.contourf(
                    plot_data[lon_dim], plot_data[lat_dim], plot_data,
                    cmap=cmap, levels=levels, extend=extend,
                )
                ax.set_title(f"{label}{suffix}", fontsize=8)
                ax.coastlines(resolution="50m", linewidth=0.5)

                cb = fig.colorbar(c, ax=ax, orientation="horizontal", pad=0.03)
                if col == 0:
                    cb.set_label(pcfg["raw_unit"], fontsize=6)
                cb.set_ticks(levels)
                cb.ax.tick_params(labelsize=6, rotation=45)

    else:
        # Single-variable 1x3 plot
        fig, axs = plt.subplots(
            1, 3, figsize=(6.5, 3.5),
            subplot_kw={"projection": ccrs.PlateCarree()}, dpi=600,
        )

        raw_data = data.isel({time_dim: 0})
        if pcfg["raw_transform"] is not None:
            raw_data = pcfg["raw_transform"](raw_data)

        configs_plot = [
            {
                "data": raw_data,
                "title": pcfg["title"],
                "cmap": pcfg["cmap"],
                "levels": pcfg["raw_levels"],
            },
            {
                "data": norm.isel({time_dim: 0}),
                "title": f"{pcfg['title']} Standardized Anomalies",
                "cmap": "balance",
                "levels": pcfg["std_levels"],
            },
            {
                "data": norm_weighted.isel({time_dim: 0}),
                "title": f"{pcfg['title']} Standardized Anomalies Weighted",
                "cmap": "balance",
                "levels": pcfg["std_levels"],
            },
        ]

        for i, cfg in enumerate(configs_plot):
            c = axs[i].contourf(
                cfg["data"][lon_dim], cfg["data"][lat_dim], cfg["data"],
                cmap=cfg["cmap"], levels=cfg["levels"], extend=pcfg["extend"],
            )
            axs[i].set_title(cfg["title"], fontsize=8)
            axs[i].coastlines(resolution="50m", linewidth=0.5)

            cb = fig.colorbar(c, ax=axs[i], orientation="horizontal", pad=0.03)
            if i == 0:
                cb.set_label(pcfg["raw_unit"], fontsize=6)
            cb.set_ticks(c.levels)
            cb.ax.tick_params(labelsize=6, rotation=45)

    plt.tight_layout()
    plt.savefig(
        f"{pcfg['fig_dir']}/{VARIABLE}_standardized_anomalies_example.png",
        dpi=600, bbox_inches="tight",
    )
    plt.close()
    print(f"Plot saved to {pcfg['fig_dir']}/")
else:
    print("Plotting skipped.")

## Extract Flash Flood Event Times & Save

In [ ]:
# Convert event hours from NYC local time to UTC
times_local = pd.to_datetime(event_hours).tz_localize("EST")
times_utc = times_local.tz_convert("UTC").tz_convert(None)

# Find intersecting times
intersect = pd.Index(times_utc).intersection(data.indexes[time_dim])
print(f"Found {len(intersect)} matching event hours out of {len(times_utc)} total.")

# Extract and save flash flood event data
out_dir = "/mnt/drive2/SOM_intermediate_files"
prefix = config["output_prefix"]

# For tp (or any variable loaded via mfdataset with a var_name), extract the variable
data_to_save = data
if "var_name" in config:
    data_to_save = data[config["var_name"]]

# Save raw data for event hours
ffe = data_to_save.sel({time_dim: intersect})
ffe.to_netcdf(f"{out_dir}/{prefix}_ffe.nc")
print(f"Saved: {out_dir}/{prefix}_ffe.nc")

# Save normalized and weighted versions if anomalies were computed
if config["compute_anomalies"]:
    norm_ffe = norm.sel({time_dim: intersect})
    norm_weighted_ffe = norm_weighted.sel({time_dim: intersect})

    norm_ffe.to_netcdf(f"{out_dir}/{prefix}_norm_ffe.nc")
    norm_weighted_ffe.to_netcdf(f"{out_dir}/{prefix}_norm_weighted_ffe.nc")
    print(f"Saved: {out_dir}/{prefix}_norm_ffe.nc")
    print(f"Saved: {out_dir}/{prefix}_norm_weighted_ffe.nc")

## Daily Averaging & Save

In [ ]:
if config["save_daily"]:
    out_dir = "/mnt/drive2/SOM_intermediate_files"
    prefix = config["output_prefix"]

    data_daily = (
        data.resample({time_dim: "1D"}, label="left", closed="left")
        .mean()
        .dropna(dim=time_dim, how="all")
    )
    data_daily.to_netcdf(f"{out_dir}/{prefix}_daily.nc")
    print(f"Saved: {out_dir}/{prefix}_daily.nc")

    if config["compute_anomalies"]:
        norm_daily = (
            norm.resample({time_dim: "1D"}, label="left", closed="left")
            .mean()
            .dropna(dim=time_dim, how="all")
        )
        norm_weighted_daily = (
            norm_weighted.resample({time_dim: "1D"}, label="left", closed="left")
            .mean()
            .dropna(dim=time_dim, how="all")
        )

        norm_daily.to_netcdf(f"{out_dir}/{prefix}_norm_daily.nc")
        norm_weighted_daily.to_netcdf(f"{out_dir}/{prefix}_norm_weighted_daily.nc")
        print(f"Saved: {out_dir}/{prefix}_norm_daily.nc")
        print(f"Saved: {out_dir}/{prefix}_norm_weighted_daily.nc")
else:
    print("Daily averaging skipped for this variable.")